# Gephi Export — BERTopic topic-similarity network

Produces `nodes.csv` and `edges.csv` for a topic-similarity graph in Gephi.  
Nodes = topics from all sources; edges = cosine similarity between topic centroids,  
sparsified to each node's top-k neighbours.

**Requirements:** same Python environment as the BERTopic notebooks  
(needs `transformers`, `torch`, `scikit-learn`, `pandas`, `numpy`, `tqdm`)

In [13]:
# ============================================================
# PARAMETERS — edit these before running
# ============================================================

# Source files: map source name -> CSV path (relative to this notebook).
# Set a path to None to skip that source.
SOURCES = {
    "news":      "../final_df/cleaned_final_df.csv",
    "talkshows": "../../subtitles/data/subs_labelled_mapped.csv",
    "kamer":     None,   # TODO: add path once tweede-kamer BERTopic is run
}

# Column names (must be the same in every source file)
TEXT_COL  = "body"         # text column used for embedding
TOPIC_COL = "topic"        # integer topic id; -1 rows are excluded automatically
LABEL_COL = "topic_label"  # human-readable label shown as Gephi node label
META_COL  = "topic_meta"   # shared meta-topic category

# Embedding model — must be the same model that was used in all BERTopic runs
EMBED_MODEL      = "GroNLP/bert-base-dutch-cased"
EMBED_BATCH_SIZE = 32

# Optional: folder to cache per-source embeddings as .npy files (speeds up reruns).
# Set to None to disable caching.
EMBED_CACHE_DIR = "embed_cache"

# --- Geometry correction (fixes anisotropy / source-register bias) ---
# Step 1: subtract each source's mean centroid from its own topic centroids.
# This removes the source-register offset so similarity reflects topic theme,
# not which corpus the topic came from.
PER_SOURCE_CENTER = True

# Step 2: subtract the projection onto the top-N global principal components
# of the pooled, already-centred centroids ("all-but-the-top" trick).
# Set to 0 to disable.
REMOVE_TOP_PCS = 1

# kNN sparsification
K          = 5      # keep each node's top-K most similar neighbours
MUTUAL_KNN = False  # if True, keep only edges where *both* nodes chose the other

# Output
OUT_NODES = "nodes.csv"
OUT_EDGES = "edges.csv"
# ============================================================

In [14]:
import pathlib, warnings
import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

## 1. Load source data

In [ ]:
NB_DIR = pathlib.Path(".").resolve()  # directory of this notebook

frames = {}  # source_name -> DataFrame
for src_name, rel_path in SOURCES.items():
    if rel_path is None:
        print(f"[SKIP] {src_name}: path is None")
        continue
    abs_path = (NB_DIR / rel_path).resolve()
    if not abs_path.exists():
        print(f"[SKIP] {src_name}: file not found at {abs_path}")
        continue
    df = pd.read_csv(abs_path)
    for col in [TEXT_COL, TOPIC_COL, LABEL_COL, META_COL]:
        if col not in df.columns:
            raise ValueError(f"{src_name}: missing required column '{col}' in {abs_path}")
    # exclude BERTopic outlier topic -1
    before = len(df)
    df = df[df[TOPIC_COL] != -1].copy()
    df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)
    frames[src_name] = df
    print(
        f"[OK] {src_name}: {len(df):,} docs "
        f"({before - len(df)} outlier rows dropped), "
        f"{df[TOPIC_COL].nunique()} topics"
    )

if not frames:
    raise RuntimeError("No source files were loaded. Check SOURCES paths above.")

## 2. Embedding model

All sources **must** use the same model so their embeddings live in the same vector space.  
The BERTopic notebooks used `GroNLP/bert-base-dutch-cased`; loading the same model here  
guarantees a shared 768-dimensional space.

In [ ]:
print(f"Loading embedding model: {EMBED_MODEL}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL)
hf_model  = AutoModel.from_pretrained(EMBED_MODEL).to(device)
hf_model.eval()
EMBED_DIM = hf_model.config.hidden_size
print(f"Embedding dimension: {EMBED_DIM}")


def embed_texts(texts, batch_size=EMBED_BATCH_SIZE):
    """Mean-pool last hidden states -> (N, EMBED_DIM) float32 ndarray."""
    all_embs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="batches", leave=False):
            batch = texts[i : i + batch_size]
            enc = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt",
            ).to(device)
            out  = hf_model(**enc)
            # mean pool over non-padding tokens
            mask = enc["attention_mask"].unsqueeze(-1).float()
            emb  = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
            all_embs.append(emb.cpu().float().numpy())
    return np.vstack(all_embs)

## 3. Compute per-topic centroids

Centroid = mean of embeddings of all documents assigned to that topic.

In [ ]:
if EMBED_CACHE_DIR:
    cache_dir = pathlib.Path(EMBED_CACHE_DIR)
    cache_dir.mkdir(parents=True, exist_ok=True)


def get_embeddings(src_name, df):
    """Return (N, EMBED_DIM) embeddings, reading from cache if available."""
    if EMBED_CACHE_DIR:
        cache_file = cache_dir / f"{src_name}_embeddings.npy"
        if cache_file.exists():
            embs = np.load(cache_file)
            if embs.shape == (len(df), EMBED_DIM):
                print(f"  [cache hit] {embs.shape} from {cache_file}")
                return embs
            print(f"  [cache stale] shape mismatch, recomputing")
    print(f"  Embedding {len(df):,} documents for {src_name!r}...")
    embs = embed_texts(df[TEXT_COL].tolist())
    if EMBED_CACHE_DIR:
        np.save(cache_file, embs)
        print(f"  Saved to {cache_file}")
    return embs


centroids   = {}  # src_name -> {topic_id: ndarray}
topic_sizes = {}  # src_name -> {topic_id: doc_count}
topic_info  = {}  # src_name -> {topic_id: (label, meta)}

for src_name, df in frames.items():
    print(f"\nProcessing {src_name!r}...")
    df = df.reset_index(drop=True)   # align positional index with embedding rows
    frames[src_name] = df

    embs = get_embeddings(src_name, df)

    # shared embedding space check: all sources must produce EMBED_DIM-dim vectors
    if embs.shape[1] != EMBED_DIM:
        raise RuntimeError(
            f"{src_name!r} produced {embs.shape[1]}-dim embeddings "
            f"but expected {EMBED_DIM}. "
            "Sources do NOT share one embedding space — "
            "check that the same EMBED_MODEL was used for all BERTopic runs."
        )

    src_centroids = {}
    src_sizes     = {}
    src_info      = {}

    for topic_id, grp in df.groupby(TOPIC_COL):
        idx = grp.index.tolist()
        src_centroids[topic_id] = embs[idx].mean(axis=0)
        src_sizes[topic_id]     = len(grp)
        src_info[topic_id]      = (grp[LABEL_COL].mode().iloc[0],
                                   grp[META_COL].mode().iloc[0])

    centroids[src_name]   = src_centroids
    topic_sizes[src_name] = src_sizes
    topic_info[src_name]  = src_info
    print(f"  {len(src_centroids)} centroids (total docs: {len(df):,})")

## 4. Build nodes table

In [ ]:
node_rows       = []
node_ids        = []   # row i in centroid matrix -> node_ids[i]
node_sources    = []   # row i -> source name (for diagnostics)
centroid_rows   = []

for src_name in centroids:
    sizes = topic_sizes[src_name]
    total = sum(sizes.values())

    for topic_id in sorted(centroids[src_name]):
        node_id     = f"{src_name}_{topic_id}"
        label, meta = topic_info[src_name][topic_id]

        node_rows.append({
            "Id":        node_id,
            "Label":     label,
            "source":    src_name,
            "metatopic": meta,
            "size":      round(sizes[topic_id] / total, 6),
        })
        node_ids.append(node_id)
        node_sources.append(src_name)
        centroid_rows.append(centroids[src_name][topic_id])

nodes_df = pd.DataFrame(node_rows)
print(f"Nodes: {len(nodes_df)}")
nodes_df.head(10)

## 5. Geometry correction

Raw BERT centroids are anisotropic: they cluster in a narrow cone dominated by
each source's register/style, so cosine similarity is ~0.98 everywhere and
within-source pairs always win the kNN race. Two corrections fix this:

1. **Per-source mean-centering** (`PER_SOURCE_CENTER`): subtract each source's
   mean centroid from its own topics. Removes the corpus-level offset so
   similarity measures topic theme, not source identity.
2. **Top-PC removal** (`REMOVE_TOP_PCS`): subtract the projection onto the top-N
   global principal components of the pooled centred matrix ("all-but-the-top").
   Removes any remaining dominant direction shared across sources.

After both steps, L2-normalise before computing cosine similarity.

In [ ]:
C = np.vstack(centroid_rows).copy()   # (n_nodes, EMBED_DIM), raw centroids
n = len(node_ids)

# 1. Per-source mean-centering
if PER_SOURCE_CENTER:
    src_index = {}   # source -> list of row indices
    for i, src in enumerate(node_sources):
        src_index.setdefault(src, []).append(i)
    for src, indices in src_index.items():
        src_mean = C[indices].mean(axis=0)
        C[indices] -= src_mean
    print(f"Per-source mean-centering applied ({len(src_index)} sources).")

# 2. Top-PC removal
if REMOVE_TOP_PCS > 0:
    n_pcs = min(REMOVE_TOP_PCS, n - 1)
    pca   = PCA(n_components=n_pcs, random_state=42)
    pca.fit(C)
    # subtract projection onto the top PCs
    C -= C @ pca.components_.T @ pca.components_
    print(
        f"Removed top {n_pcs} PC(s). "
        f"Explained variance removed: {pca.explained_variance_ratio_.sum():.1%}"
    )

# 3. L2-normalise (makes cosine similarity = dot product)
norms = np.linalg.norm(C, axis=1, keepdims=True)
norms = np.where(norms < 1e-10, 1.0, norms)   # guard zero-norm vectors
C = C / norms
print("L2-normalisation applied.")

## 6. Cosine-similarity matrix + kNN sparsification

In [ ]:
# C is already L2-normalised, so cosine_similarity = C @ C.T
sim_matrix = cosine_similarity(C)   # (n_nodes, n_nodes)
print(f"Similarity matrix: {sim_matrix.shape}")

# For kNN: zero out self before sorting
sim_knn = sim_matrix.copy()
np.fill_diagonal(sim_knn, -np.inf)

k = min(K, n - 1)   # guard: can't have more neighbours than nodes minus self

# row-wise argsort; last k columns = top-k highest similarities
topk_indices = np.argsort(sim_knn, axis=1)[:, -k:]

if MUTUAL_KNN:
    knn_sets  = [set(row) for row in topk_indices]
    raw_edges = [
        (i, j, sim_matrix[i, j])
        for i in range(n)
        for j in knn_sets[i]
        if i in knn_sets[j]
    ]
else:
    raw_edges = [
        (i, j, sim_matrix[i, j])
        for i in range(n)
        for j in topk_indices[i]
    ]

# Deduplicate undirected pairs:
# sort each pair so the lexicographically smaller Id is always Source.
deduped = {}
for i, j, w in raw_edges:
    a, b = (i, j) if node_ids[i] <= node_ids[j] else (j, i)
    key  = (a, b)
    if key not in deduped or w > deduped[key]:
        deduped[key] = w

print(f"Edges after dedup (k={k}, mutual_knn={MUTUAL_KNN}): {len(deduped)}")

## 7. Build edges table

In [ ]:
edge_rows = [
    {
        "Source": node_ids[a],
        "Target": node_ids[b],
        "Weight": round(float(w), 6),
        "Type":   "Undirected",
    }
    for (a, b), w in sorted(deduped.items())
]

edges_df = pd.DataFrame(edge_rows)
print(f"Edges: {len(edges_df)}")
edges_df.head(10)

## 8. Diagnostic summary

In [ ]:
node_src_arr = np.array(node_sources)
diag_mask    = np.eye(n, dtype=bool)
same_mask    = (node_src_arr[:, None] == node_src_arr[None, :]) & ~diag_mask
cross_mask   = (node_src_arr[:, None] != node_src_arr[None, :]) & ~diag_mask

within_sims = sim_matrix[same_mask]
cross_sims  = sim_matrix[cross_mask]

print("=" * 60)
print("SIMILARITY DISTRIBUTION (post-correction)")
print(f"  Within-source: min={within_sims.min():.4f}  "
      f"median={np.median(within_sims):.4f}  max={within_sims.max():.4f}")
if len(cross_sims):
    print(f"  Cross-source:  min={cross_sims.min():.4f}  "
          f"median={np.median(cross_sims):.4f}  max={cross_sims.max():.4f}")
else:
    print("  Cross-source:  (only one source loaded)")

cross_edge_count = sum(
    1 for (a, b) in deduped if node_sources[a] != node_sources[b]
)
print()
print("EDGE BREAKDOWN")
print(f"  Total edges:        {len(deduped)}")
print(f"  Cross-source edges: {cross_edge_count} "
      f"({100 * cross_edge_count / max(len(deduped), 1):.1f}%)")

print()
print("EDGE WEIGHT STATISTICS")
print(edges_df["Weight"].describe().round(4).to_string())

try:
    import networkx as nx
    G = nx.Graph()
    G.add_nodes_from(node_ids)
    G.add_edges_from([(node_ids[a], node_ids[b]) for a, b in deduped])
    n_comp = nx.number_connected_components(G)
    print()
    print(f"CONNECTED COMPONENTS: {n_comp}")
    if n_comp <= 10:
        for comp in nx.connected_components(G):
            srcs = {nid.split("_")[0] for nid in comp}
            print(f"  size={len(comp):3d}  sources={sorted(srcs)}")
except ImportError:
    print("\n(networkx not available — skipping component count)")
print("=" * 60)

## 9. Export

In [ ]:
nodes_df.to_csv(OUT_NODES, index=False)
edges_df.to_csv(OUT_EDGES, index=False)

print(f"Saved {len(nodes_df)} nodes  -> {OUT_NODES}")
print(f"Saved {len(edges_df)} edges  -> {OUT_EDGES}")
print()
print("Node source breakdown:")
print(nodes_df["source"].value_counts().to_string())